# 14 — Holdout easy vs hard (difficulty stratification)

**Summary:** Merges `train_ranked.csv` into the holdout, takes the **N** lowest- and **N** highest-difficulty rows (`difficulty_percentile` or `final_difficulty_score`), generates with fixed **`MAX_NEW_TOKENS`**, applies **`POSTPROCESS_METHOD`**, compares aggregate metrics, and shows two side-by-side render figures (easy / hard). Run the staged cells below in order after setting **Parameters**.

**Prerequisites:** **03** (holdout), **02** (`data/processed/train_ranked.csv`) with **id** overlap on holdout rows.

**Outputs:** CSVs under `evaluations/easy_vs_hard/<MODEL_ID>/`.


#### Colab / install


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip -q install transformers peft accelerate bitsandbytes cairosvg pillow matplotlib lxml pandas tqdm


#### Parameters


In [3]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID = 'run_1'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
OUT_ROOT = WORKFLOW_ROOT / 'evaluations' / 'easy_vs_hard'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RANKED_PATH = PROCESSED_DIR / 'train_ranked.csv'

MODEL_ID = '9c771d06b1014d22'
BASE_MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
N_EASY_HARD = 5
MAX_NEW_TOKENS = 3072
POSTPROCESS_METHOD = 'current_default_sanitizer'
N_SHOW = 2 # gallery rows per subset (<= N_EASY_HARD)

from IPython.display import display

from src.core.dataframe import choose_first_existing
from src.core.modeling_splits import load_holdout_eval
from src.eval.holdout_evaluation import (
    apply_postprocess_column,
    merge_ranked_metadata,
    predict_holdout_raw,
    score_predictions_df,
)
from src.eval.postprocess_presets import POSTPROCESS_DESCRIPTIONS
from src.eval.qualitative import make_side_by_side_render_figure
from src.inference.metrics_generation import mean_nll_over_series
from src.training.lora.eval import summarize_generation_df
from src.training.lora.modeling import load_inference_adapter
from src.training.lora.registry import load_registry, resolve_adapter_path

if not MODEL_ID:
    raise ValueError('Set MODEL_ID to a registry model_id string.')

if POSTPROCESS_METHOD not in POSTPROCESS_DESCRIPTIONS:
    raise ValueError(f'Unknown POSTPROCESS_METHOD={POSTPROCESS_METHOD!r}')

if N_SHOW > N_EASY_HARD:
    raise ValueError('N_SHOW must be <= N_EASY_HARD')

reg = load_registry(PROJECT_DIR, models_root=MODELS_ROOT)
rm = reg[reg['model_id'].astype(str) == str(MODEL_ID)]
if len(rm) != 1:
    raise ValueError(f'MODEL_ID not found uniquely in registry: {MODEL_ID!r}')
r0 = rm.iloc[0]
adapter_dir = resolve_adapter_path(PROJECT_DIR, str(r0['adapter_dir']))
if BASE_MODEL_ID is None:
    cfg = json.loads(r0['training_config_json'])
    BASE_MODEL_ID = cfg.get('base_model_id') or cfg.get('model_id')
if not BASE_MODEL_ID:
    raise ValueError('Set BASE_MODEL_ID explicitly (not in registry training_config_json).')


#### Generate and score


In [4]:
import pandas as pd

holdout_df = load_holdout_eval(WORKFLOW_ROOT)
if "id" not in holdout_df.columns:
    holdout_df = holdout_df.copy()
    holdout_df["id"] = holdout_df.index.astype(str)

ranked = pd.read_csv(RANKED_PATH)
ranked.columns = ranked.columns.str.strip()

if "id" not in ranked.columns:
    raise ValueError("train_ranked.csv must contain an 'id' column")

meta_cols = [
    c
    for c in ("difficulty_percentile", "final_difficulty_score", "difficulty_bucket")
    if c in ranked.columns
]
if not meta_cols:
    raise ValueError("train_ranked.csv has no difficulty_* columns")

ranked_sub = ranked[["id"] + meta_cols].drop_duplicates(subset=["id"])

holdout_df = holdout_df.copy()
# Avoid merge suffixes (_x/_y): drop overlapping difficulty cols from holdout first
_drop = [c for c in meta_cols if c in holdout_df.columns]
if _drop:
    holdout_df = holdout_df.drop(columns=_drop)

holdout_df["id"] = holdout_df["id"].astype(str).str.strip()
ranked_sub = ranked_sub.copy()
ranked_sub["id"] = ranked_sub["id"].astype(str).str.strip()

holdout_df = holdout_df.merge(ranked_sub, on="id", how="left")

if "difficulty_percentile" in holdout_df.columns and holdout_df["difficulty_percentile"].notna().any():
    diff_col = "difficulty_percentile"
elif "final_difficulty_score" in holdout_df.columns:
    diff_col = "final_difficulty_score"
else:
    raise ValueError(
        f"No difficulty columns after merge. Ensure {RANKED_PATH} exists (notebook 02) "
        "and holdout rows share id with ranked train rows."
    )

prompt_col = choose_first_existing(holdout_df, ["prompt", "description", "text"], "holdout")
svg_col = choose_first_existing(holdout_df, ["svg", "svg_code", "target", "label"], "holdout")
id_col = "id" if "id" in holdout_df.columns else None
if id_col is None:
    holdout_df = holdout_df.copy()
    holdout_df["id"] = holdout_df.index.astype(str)
    id_col = "id"

sub = holdout_df.dropna(subset=[diff_col]).copy()
sub = sub.sort_values(diff_col, ascending=True)
need = 2 * int(N_EASY_HARD)
if len(sub) < need:
    raise ValueError(
        f"Need at least {need} holdout rows with non-null {diff_col}; got {len(sub)}. "
        "Check id overlap with train_ranked.csv."
    )

easy_df = sub.iloc[: int(N_EASY_HARD)].reset_index(drop=True)
hard_df = sub.iloc[-int(N_EASY_HARD) :].reset_index(drop=True)

print("Easy", diff_col, "range:", easy_df[diff_col].min(), "-", easy_df[diff_col].max())
print("Hard", diff_col, "range:", hard_df[diff_col].min(), "-", hard_df[diff_col].max())

easy_raw = predict_holdout_raw(
    easy_df,
    prompt_col,
    svg_col,
    adapter_dir,
    BASE_MODEL_ID,
    max_new_tokens=int(MAX_NEW_TOKENS),
    id_col=id_col,
    show_progress=True,
    progress_desc="Easy subset generation",
)
hard_raw = predict_holdout_raw(
    hard_df,
    prompt_col,
    svg_col,
    adapter_dir,
    BASE_MODEL_ID,
    max_new_tokens=int(MAX_NEW_TOKENS),
    id_col=id_col,
    show_progress=True,
    progress_desc="Hard subset generation",
)

easy_raw = easy_raw.copy()
hard_raw = hard_raw.copy()
easy_raw["pred_svg"] = apply_postprocess_column(easy_raw, POSTPROCESS_METHOD)
hard_raw["pred_svg"] = apply_postprocess_column(hard_raw, POSTPROCESS_METHOD)

easy_scored = score_predictions_df(easy_raw, "pred_svg")
hard_scored = score_predictions_df(hard_raw, "pred_svg")
summ_easy = summarize_generation_df(easy_scored)
summ_hard = summarize_generation_df(hard_scored)

tokenizer, model = load_inference_adapter(adapter_dir, BASE_MODEL_ID)
ge = mean_nll_over_series(
    model,
    tokenizer,
    easy_raw["full_prompt"].tolist(),
    easy_raw["target_svg"].tolist(),
    show_progress=True,
    progress_desc="Gold NLL easy",
)
ce = mean_nll_over_series(
    model,
    tokenizer,
    easy_raw["full_prompt"].tolist(),
    easy_raw["raw_pred"].tolist(),
    show_progress=True,
    progress_desc="Continuation NLL easy",
)
gh = mean_nll_over_series(
    model,
    tokenizer,
    hard_raw["full_prompt"].tolist(),
    hard_raw["target_svg"].tolist(),
    show_progress=True,
    progress_desc="Gold NLL hard",
)
ch = mean_nll_over_series(
    model,
    tokenizer,
    hard_raw["full_prompt"].tolist(),
    hard_raw["raw_pred"].tolist(),
    show_progress=True,
    progress_desc="Continuation NLL hard",
)
del model, tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

summ_easy = {**summ_easy, "gold_teacher_nll_mean": ge, "continuation_nll_mean": ce}
summ_hard = {**summ_hard, "gold_teacher_nll_mean": gh, "continuation_nll_mean": ch}

compare = pd.DataFrame(
    [
        {"subset": "easy", **summ_easy},
        {"subset": "hard", **summ_hard},
    ]
)

Easy difficulty_percentile range: 0.01604 - 0.07058
Hard difficulty_percentile range: 0.95992 - 0.9899


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Easy subset generation:   0%|          | 0/5 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

#### Save outputs


In [ ]:
print('POSTPROCESS_METHOD:', POSTPROCESS_METHOD, '|', POSTPROCESS_DESCRIPTIONS.get(POSTPROCESS_METHOD, ''))
display(compare)

_out = OUT_ROOT / str(MODEL_ID)
_out.mkdir(parents=True, exist_ok=True)
easy_scored.to_csv(_out / 'easy_predictions_postprocessed.csv', index=False)
hard_scored.to_csv(_out / 'hard_predictions_postprocessed.csv', index=False)
compare.to_csv(_out / 'easy_vs_hard_summary.csv', index=False)
print('Wrote', _out)


#### Easy renders


In [ ]:
fig_e = make_side_by_side_render_figure(
    easy_scored,
    'target_svg',
    'pred_svg',
    'prompt',
    n=int(N_SHOW),
    figure_title=f'Easy {N_EASY_HARD} (showing {N_SHOW}) | MODEL_ID={MODEL_ID}',
)
plt.show()


#### Hard renders


In [ ]:
fig_h = make_side_by_side_render_figure(
    hard_scored,
    'target_svg',
    'pred_svg',
    'prompt',
    n=int(N_SHOW),
    figure_title=f'Hard {N_EASY_HARD} (showing {N_SHOW}) | MODEL_ID={MODEL_ID}',
)
plt.show()
